In [109]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import json
import os
import numpy as np
from PIL import Image

class EpisodeDataset(Dataset):
    def __init__(self, json_file, image_dir, transform=None):
        with open(json_file) as f:
            self.data = json.load(f)

        self.image_dir = image_dir
        self.transform = transform

        # Map episode names -> label index
        self.episodes = list(self.data.keys())
        self.label_map = {ep: i for i, ep in enumerate(self.episodes)}

        # Flatten into (image_path, label)
        self.samples = []
        for ep in self.episodes:
            for img_name in self.data[ep]:
                path = os.path.join(image_dir, img_name)
                self.samples.append((path, self.label_map[ep]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)


In [110]:
from torch.utils.data import Subset
import random

def episode_wise_split(dataset, train_ratio=0.8):
    train_indices = []
    test_indices = []

    # Group indices by label (episode)
    label_to_indices = {}

    for idx, (_, label) in enumerate(dataset.samples):
        label_to_indices.setdefault(label, []).append(idx)

    # Split each episode separately
    for label, indices in label_to_indices.items():
        random.shuffle(indices)

        split = int(len(indices) * train_ratio)
        train_idx = indices[:split]
        test_idx = indices[split:]

        train_indices.extend(train_idx)
        test_indices.extend(test_idx)

    return Subset(dataset, train_indices), Subset(dataset, test_indices)

In [111]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])



In [112]:
"""import zipfile

path_to_zip_file = 'randomframes.zip'
directory_to_extract_to = '/content/randomframes'

with zipfile.ZipFile(path_to_zip_file, 'r') as zip_ref:
    zip_ref.extractall(directory_to_extract_to)"""

"import zipfile\n\npath_to_zip_file = 'randomframes.zip'\ndirectory_to_extract_to = '/content/randomframes'\n\nwith zipfile.ZipFile(path_to_zip_file, 'r') as zip_ref:\n    zip_ref.extractall(directory_to_extract_to)"

In [113]:
dataset = EpisodeDataset("episodes.json", "/randomframes", transform=train_transform)

train_dataset, test_dataset = episode_wise_split(dataset, 0.8)
test_dataset.dataset.transform = test_transform

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [114]:
import torch.nn as nn
from torchvision import models

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 61)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [115]:
import torch.optim as optim
import time  # <-- add this

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(device)


cpu


In [116]:


import matplotlib.pyplot as plt

train_losses = []
test_losses = []

best_test_loss = float("inf")
patience = 3
epochs_no_improve = 0

num_epochs = 0 # you can keep this high; time will limit training

max_time = 1*60* 60
start_time = time.time()

for epoch in range(num_epochs):
    # ---- TIME CHECK ----
    elapsed_time = time.time() - start_time
    if elapsed_time > max_time:
        print(f"Stopping: reached {max_time//60} minute time limit.")
        break

    # ---- TRAIN ----
    model.train()
    running_train_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ---- VALIDATION ----
    model.eval()
    running_test_loss = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_test_loss += loss.item()

    avg_test_loss = running_test_loss / len(test_loader)
    test_losses.append(avg_test_loss)

    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Test Loss={avg_test_loss:.4f}")

    # ---- EARLY STOPPING ----
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        epochs_no_improve = 0

        # Save best model
        torch.save(model.state_dict(), "best_model.pth")

    else:
        epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print("Early stopping triggered!")
            break
    print("elaps_time",elapsed_time//60)

In [117]:
model.load_state_dict(torch.load("best_model_cuda.pth", map_location=device))
model = model.to(device)
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: '/randomframes/9n6MAgpH8z.jpg'

In [ ]:

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if total>300:
            break

print("Estimated Train Accuracy:", correct / total)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch


# Get label → episode name mapping (reverse of label_map)
idx_to_episode = {v: k for k, v in dataset.label_map.items()}

# Collect all test samples
images_list = []
labels_list = []
preds_list = []
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        images_list.append(images.cpu())
        labels_list.append(labels.cpu())
        preds_list.append(predicted.cpu())



# Concatenate
images_all = torch.cat(images_list)
labels_all = torch.cat(labels_list)
preds_all = torch.cat(preds_list)


In [ ]:
import torch.nn as nn
from torchvision import models


model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 61)

model.load_state_dict(torch.load("best_model2.pth", map_location=device))
model = model.to(device)
model.eval()
# Pick 10 random indices
indices = np.random.choice(len(images_all), 10, replace=False)

# Plot
plt.figure(figsize=(15, 6))

for i, idx in enumerate(indices):
    img = images_all[idx]

    # Convert CHW → HWC
    img = img.permute(1, 2, 0).numpy()

    true_label = idx_to_episode[labels_all[idx].item()]
    pred_label = idx_to_episode[preds_all[idx].item()]

    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(f"P: {pred_label}\nT: {true_label}", fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Load your image
img_path = "test.jpg"
img_pil = Image.open(img_path).convert("RGB")

# Apply transform for the model
img_tensor = test_transform(img_pil)
img_tensor = img_tensor.unsqueeze(0).to(device)

# Inference
model.eval() # Ensure model is in eval mode
with torch.no_grad():
    outputs = model(img_tensor)
    values, indices = torch.topk(outputs, k=2, dim=1)

    # Get the indices
    pred_label = indices[0, 0].item()
    pred_label2 = indices[0, 1].item()

    # Get the confidence scores (optional but helpful)
    # Applying softmax turns raw outputs into probabilities (0-1)
    probs = torch.nn.functional.softmax(outputs, dim=1)
    conf1 = probs[0, pred_label].item() * 100
    conf2 = probs[0, pred_label2].item() * 100

# Convert label index → episode name
pred_episode = idx_to_episode[pred_label]
pred_episode2 = idx_to_episode[pred_label2]

# --- DISPLAY RESULTS ---
plt.imshow(img_pil)
plt.axis('off')
plt.title(f"Top 1: {pred_episode} ({conf1:.2f}%)\nTop 2: {pred_episode2} ({conf2:.2f}%)")
plt.show()

print(f"Predicted episode: {pred_episode}")
print(f"2nd Predicted episode: {pred_episode2}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import math

episode_name = "S01E11 The Great Divide"
image_list = dataset.data[episode_name]

batch_size = 20  # images per figure
cols = 5
rows = 4  # 5x4 = 20 images per page

for start in range(0, len(image_list), batch_size):
    plt.figure(figsize=(15, 10))

    batch = image_list[start:start + batch_size]

    for i, img_name in enumerate(batch):
        img_path = os.path.join(dataset.image_dir, img_name)
        img = Image.open(img_path)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(img_name, fontsize=6)
        plt.axis('off')

    plt.suptitle(f"{episode_name} (images {start}–{start+len(batch)})")
    plt.tight_layout()
    plt.show()

NameError: name 'dataset' is not defined